# SQL-Based Healthcare Analytics & ML Data Preparation

**AI Internship | Milestone 02**

Query a local SQLite hospital database, join relational tables, explore appointment and prescription patterns, and prepare appointment data for machine learning using scikit-learn.

**Scope:** SQL querying, exploratory data analysis, feature engineering and preprocessing. This notebook **does not train or evaluate a machine-learning model**.



In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

## 1. Database setup

Place the original, **SQLite-compatible** `HospitalDB.sql` file beside this notebook (or under `data/HospitalDB.sql`). In Colab, upload the file to the notebook runtime before executing the setup cell.

An in-memory SQLite database is used so rerunning the notebook starts with a fresh database instead of duplicating records in an existing `.db` file. The database exists only for the current Python session.

If your SQL export contains MySQL-only syntax, it must be converted to SQLite syntax first; this notebook does not silently modify the SQL or invent missing source records.


In [ ]:
from pathlib import Path
import sqlite3

# Search common locations for the SQL dataset.
sql_candidates = [Path("HospitalDB.sql"), Path("data") / "HospitalDB.sql", Path("/content/HospitalDB.sql")]
sql_path = next((path for path in sql_candidates if path.is_file()), None)
if sql_path is None:
    raise FileNotFoundError(
        "Missing HospitalDB.sql. Put the original SQLite-compatible SQL file "
        "beside this notebook (or in data/), or upload it to Colab first."
    )

sql_script = sql_path.read_text(encoding="utf-8")
conn = sqlite3.connect(":memory:")
try:
    conn.executescript(sql_script)
    conn.commit()
except sqlite3.Error as exc:
    conn.close()
    raise RuntimeError(
        "Could not load HospitalDB.sql into SQLite. Verify the file is SQLite-compatible."
    ) from exc
print(f"Loaded SQL dataset: {sql_path}")


In [ ]:
# Display all tables available in the SQLite database

tables_query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables = pd.read_sql_query(tables_query, conn)
tables

## 2. Creating the Main Analysis Dataset

The `patients` table contains basic patient information, while clinical and operational details are stored in related tables.

To create a useful dataset for analysis, the following tables are joined:

- `patients` for patient details
- `appointments` for appointment date and reason
- `doctors` for doctor and specialty information
- `hospitals` for hospital information

A `LEFT JOIN` is used so that appointment records remain available even when some related information is missing.

In [ ]:
# Join patients, appointments, doctors, and hospitals

analysis_query = """
SELECT
    a.appointment_id,
    p.patient_id,
    p.name AS patient_name,
    p.dob,
    a.appointment_date,
    a.reason,
    d.doctor_id,
    d.name AS doctor_name,
    d.specialty,
    h.hospital_id,
    h.name AS hospital_name
FROM appointments AS a
LEFT JOIN patients AS p
    ON a.patient_id = p.patient_id
LEFT JOIN doctors AS d
    ON a.doctor_id = d.doctor_id
LEFT JOIN hospitals AS h
    ON d.hospital_id = h.hospital_id;
"""

data = pd.read_sql_query(analysis_query, conn)

data.head()

## 3. Initial Data Inspection

Before performing exploratory analysis, the combined dataset is inspected to understand:

- its number of rows and columns,
- available features,
- data types,
- missing values,
- duplicate records.

This step helps identify any cleaning or preprocessing required before analysis.

In [ ]:
# Check the size, structure, missing values, and duplicates

print("Dataset shape:", data.shape)

print("\nColumn information:")
data.info()

print("\nMissing values:")
display(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())

In [ ]:
# Convert date columns to datetime format

data["dob"] = pd.to_datetime(data["dob"])
data["appointment_date"] = pd.to_datetime(data["appointment_date"])

# Calculate patient age at the time of appointment

data["age"] = (
    data["appointment_date"].dt.year
    - data["dob"].dt.year
    - (
        (data["appointment_date"].dt.month < data["dob"].dt.month)
        | (
            (data["appointment_date"].dt.month == data["dob"].dt.month)
            & (data["appointment_date"].dt.day < data["dob"].dt.day)
        )
    )
)

data[["dob", "appointment_date", "age"]].head()

In [ ]:
# Create additional time-based features

data["appointment_month"] = data["appointment_date"].dt.month
data["appointment_month_name"] = data["appointment_date"].dt.month_name()
data["appointment_day"] = data["appointment_date"].dt.day_name()

# Create age groups for easier clinical interpretation

data["age_group"] = pd.cut(
    data["age"],
    bins=[0, 17, 35, 50, 65, 120],
    labels=["0–17", "18–35", "36–50", "51–65", "66+"],
    include_lowest=True
)

data[
    [
        "appointment_date",
        "appointment_month_name",
        "appointment_day",
        "age",
        "age_group"
    ]
].head()

## 4. Exploratory Data Analysis

The exploratory analysis focuses on patient demographics and hospital appointment patterns.

The analysis will examine:

- patient age distribution,
- most common appointment reasons,
- specialties receiving the most appointments,
- hospitals handling the most appointments,
- doctors with the highest appointment counts,
- monthly appointment trends,
- appointment patterns across age groups.

In [ ]:
data["age"].describe()

In [ ]:
# Visualize patient age distribution

plt.figure(figsize=(8, 5))
sns.histplot(data=data, x="age", bins=15, kde=True)

plt.title("Distribution of Patient Age")
plt.xlabel("Age")
plt.ylabel("Number of Appointments")
plt.show()

### Insight: Patient Age Distribution

The age distribution helps identify which age groups use hospital services most frequently. A concentration in older age groups may indicate greater healthcare needs among elderly patients, while a wider spread suggests that the hospital serves patients across different life stages.

In [ ]:
# Count appointments by reason

reason_counts = data["reason"].value_counts()

reason_counts

In [ ]:
# Visualize appointment reasons

plt.figure(figsize=(8, 5))
sns.countplot(
    data=data,
    x="reason",
    order=data["reason"].value_counts().index
)

plt.title("Appointments by Reason")
plt.xlabel("Appointment Reason")
plt.ylabel("Number of Appointments")
plt.xticks(rotation=20)
plt.show()

### Insight: Appointment Reasons

The most frequent appointment reason represents the largest source of hospital visits in this dataset. A high number of checkups or consultations may indicate routine healthcare demand, while a high number of emergency visits may suggest greater pressure on urgent-care services.

In [ ]:
# Count appointments handled by each medical specialty

specialty_counts = data["specialty"].value_counts()

specialty_counts

In [ ]:
# Visualize appointments by medical specialty

plt.figure(figsize=(10, 6))

sns.countplot(
    data=data,
    y="specialty",
    order=data["specialty"].value_counts().index
)

plt.title("Appointments by Medical Specialty")
plt.xlabel("Number of Appointments")
plt.ylabel("Medical Specialty")
plt.show()

### Insight: Medical Specialty Demand

The appointment counts show which specialties receive the highest patient demand. Specialties with larger numbers of appointments may require more doctors, longer working hours, or additional resources to manage patient flow effectively.

In [ ]:
# Count appointments handled by each hospital

hospital_counts = data["hospital_name"].value_counts()

hospital_counts

In [ ]:
# Visualize appointments handled by each hospital

plt.figure(figsize=(10, 6))

sns.countplot(
    data=data,
    y="hospital_name",
    order=data["hospital_name"].value_counts().index
)

plt.title("Appointments by Hospital")
plt.xlabel("Number of Appointments")
plt.ylabel("Hospital")
plt.show()

### Insight: Hospital Workload

Hospitals with higher appointment counts are handling a larger share of patient activity. This may indicate higher patient preference, better accessibility, more available specialists, or greater service capacity at those hospitals.

In [ ]:
# Identify doctors with the highest number of appointments

doctor_counts = (
    data.groupby(["doctor_name", "specialty"])
        .size()
        .reset_index(name="appointment_count")
        .sort_values("appointment_count", ascending=False)
)

doctor_counts.head(10)

In [ ]:
# Visualize the top 10 doctors by appointment count

top_doctors = doctor_counts.head(10)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_doctors,
    x="appointment_count",
    y="doctor_name"
)

plt.title("Top 10 Doctors by Number of Appointments")
plt.xlabel("Number of Appointments")
plt.ylabel("Doctor")
plt.show()

### Insight: Doctor Workload

The doctors with the highest appointment counts are managing the largest patient workload in the dataset. Comparing workload alongside specialty can help hospital management identify staffing needs and distribute appointments more evenly.

In [ ]:
# Count appointments by month

monthly_appointments = (
    data.groupby(["appointment_month", "appointment_month_name"])
        .size()
        .reset_index(name="appointment_count")
        .sort_values("appointment_month")
)

monthly_appointments

In [ ]:
# Visualize monthly appointment trends

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=monthly_appointments,
    x="appointment_month_name",
    y="appointment_count",
    marker="o"
)

plt.title("Monthly Appointment Trend")
plt.xlabel("Month")
plt.ylabel("Number of Appointments")
plt.xticks(rotation=45)
plt.show()

### Insight: Monthly Appointment Trend

The monthly trend shows how appointment volume changes throughout the year. Months with higher appointment counts may require additional staff and resources, while quieter months may provide better opportunities for scheduling non-urgent procedures or staff training.

In [ ]:
# Count appointments across patient age groups

age_group_counts = (
    data["age_group"]
    .value_counts()
    .sort_index()
)

age_group_counts

In [ ]:
# Visualize appointments across patient age groups

plt.figure(figsize=(8, 5))

sns.countplot(
    data=data,
    x="age_group",
    order=["0–17", "18–35", "36–50", "51–65", "66+"]
)

plt.title("Appointments by Patient Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Appointments")
plt.show()

### Insight: Patient Age Groups

The age-group distribution shows which groups account for the largest share of appointments. A higher number of visits among older patients may reflect greater healthcare needs, while a strong presence of younger patients may indicate demand for pediatric or preventive services.

In [ ]:
# Compare appointment reasons across age groups

age_reason_table = pd.crosstab(
    data["age_group"],
    data["reason"]
)

age_reason_table

In [ ]:
# Visualize appointment reasons across age groups

age_reason_table.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Appointment Reasons Across Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Number of Appointments")
plt.xticks(rotation=0)
plt.legend(title="Appointment Reason")
plt.show()

### Insight: Appointment Reasons by Age Group

This comparison helps reveal whether different age groups visit hospitals for different reasons. Any visible differences should be interpreted as patterns within this dataset only, because the data does not include confirmed diagnoses or medical outcomes.

## 5. Medication and Prescription Analysis

The prescription records are stored separately from appointment data. To analyze medication patterns, the following tables are joined:

- `prescriptions`
- `patients`
- `doctors`
- `medications`

This analysis will identify the most frequently prescribed medications and the specialties associated with prescription activity.

In [ ]:
# Join prescriptions with patients, doctors, and medications

prescription_query = """
SELECT
    pr.prescription_id,
    pr.prescribed_date,
    p.patient_id,
    p.name AS patient_name,
    d.doctor_id,
    d.name AS doctor_name,
    d.specialty,
    m.medication_id,
    m.name AS medication_name,
    m.description AS medication_description
FROM prescriptions AS pr
LEFT JOIN patients AS p
    ON pr.patient_id = p.patient_id
LEFT JOIN doctors AS d
    ON pr.doctor_id = d.doctor_id
LEFT JOIN medications AS m
    ON pr.medication_id = m.medication_id;
"""

prescriptions_data = pd.read_sql_query(prescription_query, conn)

prescriptions_data.head()

In [ ]:
# Identify the most frequently prescribed medications

medication_counts = (
    prescriptions_data["medication_name"]
    .value_counts()
    .reset_index()
)

medication_counts.columns = ["medication_name", "prescription_count"]

medication_counts.head(10)

In [ ]:
# Visualize the top 10 most frequently prescribed medications

top_medications = medication_counts.head(10)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_medications,
    x="prescription_count",
    y="medication_name"
)

plt.title("Top 10 Most Frequently Prescribed Medications")
plt.xlabel("Number of Prescriptions")
plt.ylabel("Medication")
plt.show()

### Insight: Medication Prescribing Patterns

The most frequently prescribed medications represent the highest prescription activity in this database. However, these counts should not be treated as evidence of the most common diseases because the database does not contain confirmed diagnosis information.

Prescription frequency may be influenced by patient needs, doctor specialty, medication availability, and the limited size of the dataset.

In [ ]:
# Count prescription activity by medical specialty

specialty_prescriptions = (
    prescriptions_data["specialty"]
    .value_counts()
    .reset_index()
)

specialty_prescriptions.columns = ["specialty", "prescription_count"]

specialty_prescriptions

In [ ]:
# Visualize prescription activity by medical specialty

plt.figure(figsize=(10, 6))

sns.barplot(
    data=specialty_prescriptions,
    x="prescription_count",
    y="specialty"
)

plt.title("Prescription Activity by Medical Specialty")
plt.xlabel("Number of Prescriptions")
plt.ylabel("Medical Specialty")
plt.show()

### Insight: Prescription Activity by Specialty

Specialties with higher prescription counts may be treating conditions that require more medication-based management. However, this result only reflects prescription activity in the available records and should not be interpreted as disease prevalence.

## 6. Preparing the Dataset for Machine Learning

The goal of this section is to prepare the appointment dataset for a classification model.

The target variable will be:

- `reason` — the reason for the appointment

The input features will include:

- patient age,
- appointment month,
- appointment day,
- doctor specialty,
- hospital name.

Columns such as patient name, doctor name, IDs, date of birth, and exact appointment date will not be used because they are identifiers or raw date fields rather than meaningful predictive features.

In [ ]:
# Select features and target variable

X = data[
    [
        "age",
        "appointment_month",
        "appointment_day",
        "specialty",
        "hospital_name"
    ]
].copy()

y = data["reason"].copy()

print("Feature dataset shape:", X.shape)
print("Target shape:", y.shape)

X.head()

In [ ]:
# Check missing values in the selected features and target

print("Missing values in features:")
display(X.isnull().sum())

print("\nMissing values in target:")
print(y.isnull().sum())

In [ ]:
# Split features into numerical and categorical columns

numerical_features = [
    "age",
    "appointment_month"
]

categorical_features = [
    "appointment_day",
    "specialty",
    "hospital_name"
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

In [ ]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

In [ ]:
# Create preprocessing steps for numerical and categorical features

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

preprocessor

In [ ]:
# Fit preprocessing only on training data, then transform test data.
# This prevents preprocessing parameters from learning from the test set.
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)


In [ ]:
# Display the names of the transformed features

feature_names = preprocessor.get_feature_names_out()

print("Total prepared features:", len(feature_names))
feature_names

## 7. Machine Learning Readiness

The dataset has now been prepared using the Scikit-learn workflow.

The preprocessing process included:

- separating input features and the target variable,
- splitting the data into training and testing sets,
- standardizing numerical features,
- one-hot encoding categorical features,
- fitting preprocessing steps only on the training data,
- applying the same transformations to the test data.

The resulting feature matrices are fully numerical and ready to be used with a Scikit-learn classification model.

## 8. Key Findings

The exploratory plots and tables above summarize appointment patterns by age, reason, specialty, hospital, doctor, and month, as well as prescription frequencies. Review the observed counts before making claims about clinical trends. The preprocessing stage encodes categorical features and scales numerical features **using training data only**.

These are descriptive observations within the supplied database, not evidence of population-level clinical patterns or model performance.


In [ ]:
# Reproducible numeric summary, calculated from the loaded dataset.
print("Appointment records:", len(data))
print("Prescription records:", len(prescriptions_data))
print("Mean appointment-record age:", round(data["age"].mean(), 2))
print("Most frequent appointment reasons:")
print(reason_counts.head(5).to_string())
print("Prepared training/test shapes:", X_train_prepared.shape, X_test_prepared.shape)


## 9. Conclusion and Limitations

This notebook demonstrates SQL joins, relational analysis, feature engineering, visualization and scikit-learn preprocessing for a small healthcare database. It prepares feature matrices for a future classification model, **without fitting or evaluating any classifier**.

**Limitations:** The notebook depends on the separate `HospitalDB.sql` dataset, which must be SQLite-compatible. Analyses reflect the contents and quality of that dataset; appointment frequencies do not establish disease prevalence. The workflow is educational and not intended for clinical decision-making.
